# Chapter 3 - Statistical Experiments and Significance Testing

## Chapter summary
Experiments (especially **A/B tests**) are the classic way to learn causal effects. This chapter covers how to judge whether an observed difference is real or just chance:

1. **A/B testing** - compare a treatment with a control using random assignment.
2. **Hypothesis tests** - null vs. alternative hypothesis, one-way vs. two-way tests.
3. **Resampling (permutation) tests** - build the null distribution by shuffling group labels.
4. **Statistical significance and p-values**, alpha level, Type 1 / Type 2 errors.
5. **t-tests** - the parametric counterpart of the permutation test.
6. **Multiple testing** - the more tests you run, the more false "discoveries".
7. **Degrees of freedom**, **ANOVA** and the **F-statistic** for more than two groups.
8. **Chi-square tests** and **Fisher's exact test** for counts.
9. **Multi-arm bandit** algorithms.
10. **Power and sample size**.

In [ ]:
import random
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats import power as smpower
from statsmodels.stats.proportion import proportion_effectsize
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/"

def get_data(name):
    """Download a dataset of the book (once) into ./data and return its path."""
    path = Path("data") / name
    path.parent.mkdir(exist_ok=True)
    if not path.exists():
        urllib.request.urlretrieve(DATA_URL + name, path)
    return path

sns.set_style("whitegrid")
random.seed(1); np.random.seed(1)

## 1. Resampling: the permutation test

**Theory.**
* **Null hypothesis (H0)**: any difference between groups is due to chance. **Alternative (H1)**: the difference is real.
* **Permutation test**: combine the results of both groups, shuffle them, randomly split them into groups of the original sizes, and compute the difference in the statistic. Repeating this many times produces the distribution of differences *under H0*. If the observed difference lies in the extreme tail of this distribution, it is unlikely to be chance.
* The **p-value** is the probability of getting a result as extreme as (or more extreme than) the observed one, *given that H0 is true*. It is **not** the probability that H0 is true.

Example: **web stickiness** - time (seconds) visitors spent on page A vs. page B.

In [ ]:
session_times = pd.read_csv(get_data("web_page_data.csv"))
session_times.Time = 100 * session_times.Time   # convert to seconds as in the book
session_times.boxplot(by="Page", column="Time", figsize=(4, 4))
plt.suptitle(""); plt.title(""); plt.ylabel("Time (seconds)"); plt.show()

mean_a = session_times[session_times.Page == "Page A"].Time.mean()
mean_b = session_times[session_times.Page == "Page B"].Time.mean()
print("Observed difference (B - A):", mean_b - mean_a)

In [ ]:
def perm_fun(x, nA, nB):
    n = nA + nB
    idx_B = set(random.sample(range(n), nB))
    idx_A = set(range(n)) - idx_B
    return x.iloc[list(idx_B)].mean() - x.iloc[list(idx_A)].mean()

nA = session_times[session_times.Page == "Page A"].shape[0]
nB = session_times[session_times.Page == "Page B"].shape[0]
print("Example permuted difference:", perm_fun(session_times.Time, nA, nB))

perm_diffs = [perm_fun(session_times.Time, nA, nB) for _ in range(1000)]

fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(perm_diffs, bins=11, rwidth=0.9)
ax.axvline(x=mean_b - mean_a, color="black", lw=2)
ax.text(mean_b - mean_a, 190, "Observed\ndifference", bbox={"facecolor": "white"})
ax.set_xlabel("Session time differences (in seconds)"); ax.set_ylabel("Frequency")
plt.tight_layout(); plt.show()

In [ ]:
p_perm = np.mean([diff > (mean_b - mean_a) for diff in perm_diffs])
print(f"Permutation p-value (one-sided): {p_perm:.3f}")

The p-value is large (~0.12-0.15), so a difference this big arises by chance often - **not statistically significant**.
(A variant, the *exhaustive/bootstrap permutation test*, enumerates all splits or samples with replacement - results are similar.)

### Statistical significance, p-values and errors

**Statistical significance** is how statisticians decide whether an experimental result is more extreme than what chance could plausibly produce. The logic is:

1. Assume the **null hypothesis** (H0: "no real difference; the treatment does nothing") is true. Under H0 the observed difference between A and B is purely random variation, so group labels are interchangeable - which is exactly what the permutation test exploits.
2. Generate the distribution of the statistic under H0 (the histogram above).
3. Locate the observed result in that distribution. The **p-value** is the proportion of null-distribution values at least as extreme as the observed one.
4. Compare the p-value with a pre-chosen **alpha level** (the significance threshold, commonly 0.05). If $p<\alpha$ the result is called *statistically significant* and H0 is rejected.

**Reading a p-value correctly.** A p-value answers: *"If H0 were true, how often would random chance alone produce a result this extreme?"* It is **not** the probability that H0 is true, nor the probability that the result is due to chance. A p-value of 0.12 means a difference this large would appear by luck about 12% of the time even if the pages were identical - so we don't have enough evidence to claim page B is better. Conversely, a very small p-value does not mean the effect is *large or important*: with enough data, trivially small differences become "significant". Always look at the effect size and a confidence interval as well, and remember that the 0.05 cutoff is a convention, not a law of nature. (In 2016 the American Statistical Association warned against basing conclusions solely on p < 0.05.)

**Type 1 and Type 2 errors.** Any decision can be wrong in two ways:

| | H0 actually true | H0 actually false |
|---|---|---|
| **Reject H0** | **Type 1 error** (false positive, probability $\alpha$) | correct (probability = *power*) |
| **Do not reject H0** | correct | **Type 2 error** (false negative, probability $\beta$) |

Lowering $\alpha$ reduces false positives but increases false negatives, unless the sample size grows.

**One-way vs. two-way tests.** A *one-way (one-tailed)* test counts extreme results in only one direction (e.g. "B is *better* than A") and puts all the alpha in one tail of the null distribution; a *two-way (two-tailed)* test counts extremes in both directions ("B is *different* from A") and splits alpha across both tails, which roughly doubles the p-value. Decide the direction **before** seeing the data - choosing it afterwards to get a smaller p-value is a form of cheating. The permutation p-value above is one-way (we only counted differences larger than the observed one).

## 2. t-test

**Theory.** Before computers, tests relied on theoretical distributions. The **two-sample t-test** standardizes the difference in means:

$$ t = \frac{\bar{x}_B-\bar{x}_A}{\sqrt{s_A^2/n_A + s_B^2/n_B}} $$

and compares it to a t-distribution (Welch's version doesn't assume equal variances). It gives a p-value close to the permutation test when the data are roughly normal or samples are large. A **one-way** test counts only one direction of difference; a **two-way** (two-sided) test counts both.

In [ ]:
res = stats.ttest_ind(session_times[session_times.Page == "Page A"].Time,
                      session_times[session_times.Page == "Page B"].Time,
                      equal_var=False)
print(f"t-statistic = {res.statistic:.3f}, two-sided p = {res.pvalue:.4f}, one-sided p = {res.pvalue / 2:.4f}")

## 3. Multiple testing

**Theory.** If you test 20 *true-null* hypotheses at $\alpha=0.05$, you expect about one false positive (**alpha inflation**). With $m$ independent tests, $P(\text{at least one false positive}) = 1-(1-\alpha)^m$. Corrections include **Bonferroni** (use $\alpha/m$) and the **false discovery rate** (Benjamini-Hochberg). Also beware of *data dredging* and *p-hacking*: searching until something is "significant".

In [ ]:
m = np.arange(1, 51)
plt.figure(figsize=(5, 3.5))
plt.plot(m, 1 - (1 - 0.05) ** m)
plt.xlabel("Number of tests"); plt.ylabel("P(at least 1 false positive)")
plt.title("Alpha inflation at 0.05"); plt.show()

# Bonferroni / Benjamini-Hochberg on simulated p-values from 20 null tests
pvals = np.random.uniform(size=20)
from statsmodels.stats.multitest import multipletests
for method in ("bonferroni", "fdr_bh"):
    rej, adj, *_ = multipletests(pvals, alpha=0.05, method=method)
    print(f"{method:11s}: raw rejections={np.sum(pvals < 0.05)}, adjusted rejections={rej.sum()}")

### Degrees of freedom

**Degrees of freedom (df)** is the number of values in a calculation that are free to vary once certain constraints are fixed. Example: if you know the mean of $n$ numbers, only $n-1$ of them can be chosen freely - the last is determined, because the deviations from the mean must sum to zero. So the sample variance divides by $n-1$, not $n$:

$$ s^2=\frac{\sum_{i=1}^{n}(x_i-\bar{x})^2}{n-1}. $$

Dividing by $n$ would *underestimate* the true population variance because the sample mean is, by construction, closer to the sample points than the true mean is. Degrees of freedom matter to data scientists because they determine the **shape of the reference distributions** (t, chi-square, F) used for p-values and confidence intervals; in regression the df are $n-p-1$ ($p$ predictors plus the intercept), and in a chi-square test of an $r\times c$ table they are $(r-1)(c-1)$. In large datasets df rarely matters numerically - but **factor variables with many levels** (one-hot encoded) can consume many degrees of freedom and cause **multicollinearity** if one reference level isn't dropped.

Simulation: dividing by $n$ is biased, dividing by $n-1$ is not.

In [ ]:
rng_df = np.random.default_rng(0)
samples = rng_df.normal(0, 1, size=(20000, 5))      # true variance = 1, tiny samples of n = 5
print("Average variance estimate, divide by n   :", samples.var(axis=1, ddof=0).mean().round(3))
print("Average variance estimate, divide by n-1 :", samples.var(axis=1, ddof=1).mean().round(3))

## 4. ANOVA (Analysis of Variance) and the F-statistic

**Theory.** Comparing more than two groups pairwise raises the multiple-testing problem. **ANOVA** performs a single **omnibus test** of whether *all* group means are equal.
* **Permutation approach**: compute the *variance of the group means*, then see how that compares with variance of means from shuffled data.
* **F-statistic**: ratio of the between-group variance (treatment mean square) to within-group variance (residual mean square). A large F means group means differ more than chance would explain. The F-statistic follows an F-distribution (Chapter 2) with $(k-1,\ N-k)$ degrees of freedom.

Example: page-load time (in seconds) for four web pages.

In [ ]:
four_sessions = pd.read_csv(get_data("four_sessions.csv"))
four_sessions.boxplot(by="Page", column="Time", figsize=(5, 4))
plt.suptitle(""); plt.title(""); plt.ylabel("Time (seconds)"); plt.show()

observed_variance = four_sessions.groupby("Page").mean().var().iloc[0]
print("Observed variance of group means:", observed_variance)

def perm_test(df):
    df = df.copy()
    df["Time"] = np.random.permutation(df["Time"].values)
    return df.groupby("Page").mean().var().iloc[0]

perm_variance = [perm_test(four_sessions) for _ in range(3000)]
print("Permutation p-value:", np.mean([v > observed_variance for v in perm_variance]))

In [ ]:
model = smf.ols("Time ~ Page", data=four_sessions).fit()
print(sm.stats.anova_lm(model))
print("scipy f_oneway:", stats.f_oneway(*[g.Time.values for _, g in four_sessions.groupby("Page")]))

Both approaches give a p-value around 0.08 - weak evidence against equal means at the 5% level.

## 5. Chi-square test

**Theory.** For **count data** (e.g. click rates for three headlines), the **chi-square test** checks whether observed counts are consistent with an independence/"no difference" null hypothesis.
1. Compute **expected** counts under H0.
2. Pearson residual $R=\frac{O-E}{\sqrt{E}}$ for each cell.
3. $\chi^2=\sum R^2$. Compare with a chi-square distribution with $(r-1)(c-1)$ degrees of freedom (or build a resampling distribution).

**Fisher's exact test** gives exact p-values for small counts (2x2 tables).

In [ ]:
clicks = pd.DataFrame({"Headline A": [14, 986], "Headline B": [8, 992], "Headline C": [12, 988]},
                      index=["Click", "No-click"])
print(clicks)

row_average = clicks.mean(axis=1)
expected = pd.DataFrame({h: row_average for h in clicks.columns})
chi2_obs = (((clicks - expected) ** 2) / expected).values.sum()
print("Observed chi2:", chi2_obs)

# resampling distribution
box = [1] * 34 + [0] * 2966
expected_arr = expected.values

def perm_fun_chi(box):
    random.shuffle(box)
    a = sum(box[0:1000]); b = sum(box[1000:2000]); c = sum(box[2000:3000])
    obs = np.array([[a, b, c], [1000 - a, 1000 - b, 1000 - c]])
    return (((obs - expected_arr) ** 2) / expected_arr).sum()

perm_chi2 = [perm_fun_chi(box) for _ in range(2000)]
print("Resampled p-value:", np.mean([c > chi2_obs for c in perm_chi2]))

chi2_stat, p, dof, _ = stats.chi2_contingency(clicks)
print(f"scipy chi2={chi2_stat:.3f}, p={p:.3f}, dof={dof}")

In [ ]:
x = np.arange(1, 30, 0.1)
plt.figure(figsize=(5, 3.5))
for d in (1, 2, 5):
    plt.plot(x, stats.chi2.pdf(x, df=d), label=f"df={d}")
plt.legend(); plt.title("Chi-square distribution"); plt.show()

# Fisher's exact test (2x2): clicks vs. no-clicks for headline A vs. B
odds, p = stats.fisher_exact([[14, 986], [8, 992]])
print(f"Fisher exact: odds ratio={odds:.2f}, p={p:.3f}")

## 6. Multi-arm bandit algorithm

**Theory.** A traditional A/B test fixes the sample size in advance. A **multi-arm bandit** adapts: it gradually allocates more traffic to the better-performing arm (**exploitation**) while still **exploring** the others. A simple approach is **epsilon-greedy**: with probability $\epsilon$ show a random arm, otherwise show the arm with the best observed rate. **Thompson sampling** draws each arm's success rate from its Beta posterior and plays the highest draw. Bandits reduce the cost of showing a poor variant and are well suited to many options or continuous optimisation.

In [ ]:
true_rates = {"A": 0.020, "B": 0.032, "C": 0.025}
wins = {k: 1 for k in true_rates}; losses = {k: 1 for k in true_rates}
plays = {k: 0 for k in true_rates}
rng = np.random.default_rng(0)
for _ in range(20000):
    draws = {k: rng.beta(wins[k], losses[k]) for k in true_rates}   # Thompson sampling
    arm = max(draws, key=draws.get)
    plays[arm] += 1
    if rng.random() < true_rates[arm]:
        wins[arm] += 1
    else:
        losses[arm] += 1
print("Impressions per arm (Thompson sampling):", plays)

## 7. Power and sample size

**Theory.**
* **Type 1 error**: rejecting H0 when it is true (probability $\alpha$). **Type 2 error**: failing to reject a false H0.
* **Power** = $1-P(\text{Type 2})$ = probability of detecting a real effect of a given size.
* Power depends on the **effect size**, **sample size**, **significance level** and variability. Choosing the sample size before an experiment (power analysis) prevents underpowered, inconclusive tests.
* For proportions, Cohen's $h = 2\arcsin\sqrt{p_1}-2\arcsin\sqrt{p_2}$ is the effect size.

In [ ]:
effect_size = proportion_effectsize(0.0121, 0.011)
analysis = smpower.NormalIndPower()
n = analysis.solve_power(effect_size=effect_size, alpha=0.05, power=0.8, alternative="larger")
print(f"Sample size per group to detect 1.1% -> 1.21% conversion with 80% power: {n:,.0f}")

effect_size = proportion_effectsize(0.0165, 0.011)
n = analysis.solve_power(effect_size=effect_size, alpha=0.05, power=0.8, alternative="larger")
print(f"Sample size per group to detect 1.1% -> 1.65% conversion with 80% power: {n:,.0f}")

## Key takeaways
* Randomisation + a control group lets us attribute differences to the treatment (A/B test).
* A **p-value** measures how surprising the data are under H0; it is not the probability H0 is true, and "significant" does not mean "important".
* **Permutation tests** need no distributional assumptions; **t-tests / ANOVA / chi-square** are the classic parametric versions and agree when assumptions hold.
* Correct for **multiple comparisons**; use **power analysis** to choose sample size; **bandits** adapt allocation during the experiment.